# Chương 16: Tính cứng cáp và Quyền riêng tư trong AI

Notebook này triển khai thực nghiệm 3 trụ cột an ninh và bảo mật AI từ các phép toán cơ bản bằng NumPy:
1. **Tấn công đối kháng FGSM (Fast Gradient Sign Method) và PGD (Projected Gradient Descent)**: Đo lường mức độ suy giảm độ chính xác và trực quan hóa các điểm dữ liệu đối kháng vượt qua biên quyết định.
2. **Tấn công suy luận thành viên (Membership Inference Attack - MIA)**: Khai thác hiện tượng quá khớp (overfitting) để phân biệt dữ liệu thuộc tập huấn luyện (members) và dữ liệu ngoài (non-members).
3. **Huấn luyện bảo vệ quyền riêng tư DP-SGD (Differential Privacy)**: Cài đặt cắt tỉa gradient chuẩn $L_2$ (Gradient Clipping) và thêm nhiễu Gauss, đánh giá sự đánh đổi giữa mức độ bảo mật và độ chính xác.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("Môi trường sẵn sàng: NumPy", np.__version__)

## 1. Tấn công đối kháng Hộp trắng: FGSM và PGD

### Công thức toán học
- **FGSM (Goodfellow et al., 2014)**:
$$x_{\text{adv}} = x + \epsilon \cdot \text{sign}(\nabla_x \mathcal{L}(x, y))$$
- **PGD (Madry et al., 2018)**:
$$x^{(t+1)} = \text{clip}\left(x^{(t)} + \alpha \cdot \text{sign}(\nabla_x \mathcal{L}(x^{(t)}, y)), x - \epsilon, x + \epsilon\right)$$

In [ ]:
# Xây dựng mô hình phân loại Logistic Regression 2 chiều
class LogisticClassifier:
    def __init__(self, in_dim=2):
        self.w = np.random.randn(in_dim) * 0.5
        self.b = 0.0
        
    def forward(self, X):
        z = X @ self.w + self.b
        return 1.0 / (1.0 + np.exp(-np.clip(z, -15, 15)))
    
    def loss(self, X, y):
        preds = self.forward(X)
        eps = 1e-12
        return -np.mean(y * np.log(preds + eps) + (1 - y) * np.log(1 - preds + eps))
    
    def grad_x(self, x, y):
        """Đạo hàm của hàm mất mát Cross-Entropy theo đầu vào x"""
        pred = self.forward(x.reshape(1, -1))[0]
        # dL/dz = pred - y
        # dz/dx = w
        return (pred - y) * self.w

# Tạo dữ liệu phân loại 2 cụm nhị phân tách biệt rõ ràng
N_pts = 150
X_clean = np.vstack([
    np.random.randn(N_pts, 2) * 0.4 + np.array([-1.0, -1.0]),
    np.random.randn(N_pts, 2) * 0.4 + np.array([1.0, 1.0])
])
y_clean = np.array([0] * N_pts + [1] * N_pts)

# Huấn luyện mô hình chuẩn
model = LogisticClassifier(in_dim=2)
lr = 0.1
for _ in range(200):
    preds = model.forward(X_clean)
    grad_w = X_clean.T @ (preds - y_clean) / len(y_clean)
    grad_b = np.mean(preds - y_clean)
    model.w -= lr * grad_w
    model.b -= lr * grad_b

acc_clean = np.mean((model.forward(X_clean) >= 0.5) == y_clean) * 100
print(f"Độ chính xác trên dữ liệu sạch ban đầu: {acc_clean:.2f}%")

def attack_fgsm(model, X, y, epsilon=0.25):
    X_adv = np.copy(X)
    for i in range(len(X)):
        g_x = model.grad_x(X[i], y[i])
        X_adv[i] = X[i] + epsilon * np.sign(g_x)
    return X_adv

def attack_pgd(model, X, y, epsilon=0.25, alpha=0.05, num_iter=10):
    X_adv = X + np.random.uniform(-epsilon, epsilon, X.shape)
    for _ in range(num_iter):
        for i in range(len(X)):
            g_x = model.grad_x(X_adv[i], y[i])
            X_adv[i] = X_adv[i] + alpha * np.sign(g_x)
            # Phép chiếu L-infinity
            X_adv[i] = np.clip(X_adv[i], X[i] - epsilon, X[i] + epsilon)
    return X_adv

# Thực hiện tấn công với các mức epsilon
eps_vals = [0.0, 0.1, 0.2, 0.3, 0.4]
fgsm_accs, pgd_accs = [], []

for eps in eps_vals:
    X_fgsm = attack_fgsm(model, X_clean, y_clean, epsilon=eps)
    X_pgd = attack_pgd(model, X_clean, y_clean, epsilon=eps, alpha=0.05, num_iter=15)
    fgsm_accs.append(np.mean((model.forward(X_fgsm) >= 0.5) == y_clean) * 100)
    pgd_accs.append(np.mean((model.forward(X_pgd) >= 0.5) == y_clean) * 100)

plt.figure(figsize=(7, 4))
plt.plot(eps_vals, fgsm_accs, 'o--', color="orange", lw=2, label="FGSM Attack (1-step)")
plt.plot(eps_vals, pgd_accs, 's-', color="crimson", lw=2, label="PGD Attack (15-step)")
plt.title("Độ sụp giảm chính xác của bộ phân loại dưới tấn công đối kháng")
plt.xlabel("Biên độ nhiễu Epsilon")
plt.ylabel("Độ chính xác (%)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Mô phỏng tấn công suy luận thành viên (Membership Inference Attack - MIA)

Mô hình quá khớp (Overfitted) có độ mất mát trên tập Train thấp hơn nhiều so với tập Test chưa nhìn thấy. Kẻ tấn công thiết lập một ngưỡng mất mát $\tau$: nếu $\mathcal{L}(x, y) \le \tau$, kết luận mẫu $x$ thuộc tập huấn luyện (Member).

In [ ]:
# Tạo tập huấn luyện (In-sample/Member) và tập kiểm thử (Out-of-sample/Non-member)
n_member = 100
X_train_mia = np.random.randn(n_member, 10)
y_train_mia = np.random.randint(0, 2, size=n_member)

X_test_mia = np.random.randn(n_member, 10)
y_test_mia = np.random.randint(0, 2, size=n_member)

# Mô hình ghi nhớ quá mức (Overfitting Model)
W_overfit = np.random.randn(10, 1) * 2.0

# Giả lập phân phối tổn thất: Member có loss trung bình rất thấp, Non-member có loss cao
losses_members = np.abs(np.random.normal(0.1, 0.05, size=n_member))
losses_non_members = np.abs(np.random.normal(0.8, 0.25, size=n_member))

# Xây dựng bộ phân loại MIA bằng ngưỡng threshold
thresholds = np.linspace(0.0, 1.2, 100)
tpr_list, fpr_list = [], []

for th in thresholds:
    # Dự đoán là Member nếu Loss <= threshold
    tp = np.sum(losses_members <= th)
    fp = np.sum(losses_non_members <= th)
    tpr_list.append(tp / n_member)
    fpr_list.append(fp / n_member)

plt.figure(figsize=(6, 4))
plt.plot(fpr_list, tpr_list, color="darkviolet", lw=2, label="Đường cong ROC của MIA")
plt.plot([0, 1], [0, 1], 'k--', label="Đoán mò ngẫu nhiên (AUC=0.5)")
plt.title("Năng lực tấn công suy luận thành viên (MIA)")
plt.xlabel("Tỷ lệ dương tính giả (FPR)")
plt.ylabel("Tỷ lệ dương tính thật (TPR)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

print(f"MIA phát hiện thành viên với khả năng phân tách rõ ràng do hiện tượng quá khớp dữ liệu!")

## 3. Cài đặt DP-SGD (Differential Privacy SGD) từ đầu

### Các bước cốt lõi
1. Cắt tỉa gradient từng mẫu theo chuẩn $L_2$: $\bar{g}_i = \frac{g_i}{\max(1, \|g_i\|_2 / C)}$.
2. Thêm nhiễu Gauss: $\tilde{g} = \frac{1}{B} \left(\sum_{i=1}^B \bar{g}_i + \mathcal{N}(0, \sigma^2 C^2 I)\right)$.

In [ ]:
def train_dp_sgd(X, y, clipping_C=1.0, noise_sigma=0.5, epochs=50, lr=0.1, batch_size=32):
    N, D = X.shape
    w = np.zeros(D)
    b = 0.0
    
    for epoch in range(epochs):
        indices = np.random.permutation(N)
        for start in range(0, N, batch_size):
            batch_idx = indices[start:start+batch_size]
            X_b, y_b = X[batch_idx], y[batch_idx]
            B = len(X_b)
            
            # 1. Tính gradient trên từng mẫu riêng biệt (Per-sample gradients)
            preds_b = 1.0 / (1.0 + np.exp(-np.clip(X_b @ w + b, -15, 15)))
            errors = preds_b - y_b
            
            per_sample_grad_w = X_b * errors[:, None]  # (B, D)
            per_sample_grad_b = errors                 # (B,)
            
            # 2. Cắt tỉa gradient từng mẫu theo ngưỡng chuẩn L2 (Clipping)
            norms_w = np.linalg.norm(per_sample_grad_w, axis=1, keepdims=True)
            clip_factors = np.maximum(1.0, norms_w / clipping_C)
            clipped_grad_w = per_sample_grad_w / clip_factors
            
            # 3. Tổng hợp và thêm nhiễu Gauss (Noise Addition)
            noise_w = np.random.randn(D) * (noise_sigma * clipping_C)
            noisy_grad_w = (np.sum(clipped_grad_w, axis=0) + noise_w) / B
            
            # Cập nhật trọng số
            w -= lr * noisy_grad_w
            b -= lr * np.mean(errors)
            
    return w, b

# Đánh giá độ chính xác khi thay đổi cường độ nhiễu riêng tư sigma
sigmas = [0.0, 0.2, 0.5, 1.0, 2.0]
accuracies = []

for s in sigmas:
    w_dp, b_dp = train_dp_sgd(X_clean, y_clean, clipping_C=1.0, noise_sigma=s, epochs=40, lr=0.1)
    preds = 1.0 / (1.0 + np.exp(-np.clip(X_clean @ w_dp + b_dp, -15, 15)))
    acc = np.mean((preds >= 0.5) == y_clean) * 100
    accuracies.append(acc)
    print(f"Noise Sigma = {s:3.1f} | Độ chính xác mô hình DP: {acc:5.2f}%")

plt.figure(figsize=(6, 4))
plt.plot(sigmas, accuracies, 'd-', color="teal", lw=2)
plt.title("Sự đánh đổi giữa Quyền riêng tư (Độ lớn nhiễu $\sigma$) và Độ chính xác")
plt.xlabel("Hệ số nhiễu Gaussian $\sigma$")
plt.ylabel("Độ chính xác (%)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("Hoàn tất toàn bộ thực hành Chapter 16: FGSM/PGD, MIA, và DP-SGD!")